# 13 — Grafik Interaktif dengan Plotly

Dengan Menggunakan **Plotly** kita bisa membuat grafik **interaktif**: arahkan kursor untuk melihat detail (hover), zoom, dan geser. Sangat berguna untuk eksplorasi dan dashboard. Kita pakai data nyata **World Happiness Report 2019**.

> 🎯 **Tujuan Pembelajaran**
> - Mengenal **Plotly Express** (`plotly.express` / `px`) untuk grafik cepat & interaktif.
> - Membuat **scatter interaktif** dengan ukuran titik & info hover.
> - Membuat **bar chart** 10 negara paling bahagia.
> - Membuat **choropleth** (peta dunia berwarna) berdasarkan skor kebahagiaan.



## 📊 Tentang Dataset

**World Happiness Report 2019** memeringkat 156 negara berdasarkan skor kebahagiaan, beserta
faktor-faktor yang menjelaskannya (ekonomi, dukungan sosial, kesehatan, kebebasan, dll).

- **Sumber:** [Kaggle — unsdsn/world-happiness](https://www.kaggle.com/datasets/unsdsn/world-happiness)
- **File:** `2019.csv` — 156 baris × 9 kolom

| Kolom | Arti |
|-------|------|
| `Overall rank` | Peringkat kebahagiaan |
| `Country or region` | Nama negara |
| `Score` | Skor kebahagiaan (0–10) |
| `GDP per capita` | Kontribusi PDB per kapita |
| `Social support` | Dukungan sosial |
| `Healthy life expectancy` | Harapan hidup sehat |
| `Freedom to make life choices` | Kebebasan memilih jalan hidup |
| `Generosity` | Kedermawanan |
| `Perceptions of corruption` | Persepsi korupsi |

## 1.Impor Library


- **pandas** → mengolah data tabel.
- **plotly.express** (`px`) → membuat grafik interaktif dengan sedikit kode.
- **plotly.io** (`pio`) → mengatur *renderer* agar grafik ter-embed saat notebook dijalankan.


> ℹ️ Kita set `pio.renderers.default = "notebook"` dan menampilkan figur dengan menaruh objek
> `fig` sebagai baris terakhir cell — bukan `fig.show()` — supaya grafik tersimpan di output.

In [26]:
import pandas as pd

import plotly.express as px
import plotly.io as pio

pio.renderers.default = "colab"

### Import Dataset dari Kaggle

In [2]:
!pip install -q kaggle

In [3]:
!kaggle datasets download -d unsdsn/world-happiness

Dataset URL: https://www.kaggle.com/datasets/unsdsn/world-happiness
License(s): CC0-1.0
100% 36.8k/36.8k [00:00<00:00, 37.5MB/s]



In [4]:
!unzip -qq world-happiness.zip

## 2.Eksplorasi awal Dataset

In [5]:
df = pd.read_csv("2019.csv")
df.head()

,Overall rank,Country or region,Score,GDP per capita,Social support,Healthy life expectancy,Freedom to make life choices,Generosity,Perceptions of corruption
0,1,Finland,7.769,1.340,1.587,0.986,0.596,0.153,0.393
1,2,Denmark,7.600,1.383,1.573,0.996,0.592,0.252,0.410
2,3,Norway,7.554,1.488,1.582,1.028,0.603,0.271,0.341
3,4,Iceland,7.494,1.380,1.624,1.026,0.591,0.354,0.118
4,5,Netherlands,7.488,1.396,1.522,0.999,0.557,0.322,0.298


Interpretasi :   
- Negara diurutkan berdasarkan Rank Teratas dari Score World Happiness, dengan Finland menjadi Rank 1 dalam Score Happiness Terbaik
- Kolom Feature ( GDP, Social Support, Healthy life, dll) menunjukkan kontribusi feature terhadap Score

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 156 entries, 0 to 155
Data columns (total 9 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Overall rank                  156 non-null    int64  
 1   Country or region             156 non-null    object 
 2   Score                         156 non-null    float64
 3   GDP per capita                156 non-null    float64
 4   Social support                156 non-null    float64
 5   Healthy life expectancy       156 non-null    float64
 6   Freedom to make life choices  156 non-null    float64
 7   Generosity                    156 non-null    float64
 8   Perceptions of corruption     156 non-null    float64
dtypes: float64(7), int64(1), object(1)
memory usage: 11.1+ KB


In [8]:
df.isnull().sum().sum()

np.int64(0)

Interpretasi :   
-  Hampir semua kolom berisi nilai numerik
-  1 fitur berisi data kategorik yaitu kolom Negara
- data terdiri dari 9 Kolom dan 156 baris, dan setiap baris mewakili entitas sebuah negara
- Tidak ada missing value dari dataset

In [7]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Overall rank,156.0,78.500000,45.177428,1.000,39.75000,78.5000,117.25000,156.000
Score,156.0,5.407096,1.113120,2.853,4.54450,5.3795,6.18450,7.769
GDP per capita,156.0,0.905147,0.398389,0.000,0.60275,0.9600,1.23250,1.684
Social support,156.0,1.208814,0.299191,0.000,1.05575,1.2715,1.45250,1.624
Healthy life expectancy,156.0,0.725244,0.242124,0.000,0.54775,0.7890,0.88175,1.141
Freedom to make life choices,156.0,0.392571,0.143289,0.000,0.30800,0.4170,0.50725,0.631
Generosity,156.0,0.184846,0.095254,0.000,0.10875,0.1775,0.24825,0.566
Perceptions of corruption,156.0,0.110603,0.094538,0.000,0.04700,0.0855,0.14125,0.453


Interpretasi :   
- Range Score ada di angka 2.85 - 7.78, dengan rata rata 5.4
- GDP per Kapita dan Social Support punya nilai rata rat cukup tinggi, asumsi sementara keduanya menyumbang besar pengaruh pada score di banyak negara

## 3.Scatter Interaktif : GDP vs Happiness Score

### Grafik 1 - Scatter GDP vs Score Happiness

In [35]:
fig = px.scatter(
    df,
    x="GDP per capita",
    y="Score",
    size="Social support",
    color="Healthy life expectancy",
    hover_name="Country or region",
    size_max=25,
    color_continuous_scale="Viridis",
    title="GDP per Capita vs Happiness Score Data 2019",
    labels={
        "GDP per capita": "GDP per Capita",
        "Score": "Happiness Score",
        "Healthy life expectancy": "Healthy Life Expectancy",
    },
)
fig.update_layout(width=800, height=500)
fig


Interpretasi :   
- Dapat dilihat ada trend naik yang jelas
- Makin tinggi GDP per Kapita, Makin tinggi Happiness Score sebuah negara
- Titik yang semakin besar dan terang juga megindikasikan Social Support dan Healthy Life Expentancy mempengaruhi Happines Score sebuah negara
- Kita bisa arahkan cursor kita untuk melihat nama negara disertai score dan value value pendukung lainnya di setiap titik pada canvas

## 4.Bar Chart: 10 Negara paling Bahagia

Gunakan 10 Negara dengan skor tertinngi , lali tampilkan sebagai Horizontal Bar

### Grafik 2 - Bar 10 Negara paling Bahagia

In [36]:
top_10 = df.nsmallest(10, "Overall rank").sort_values("Score")

In [41]:
fig = px.bar(
    top_10,
    x="Score",
    y="Country or region",
    color="Score",
    color_continuous_scale="Viridis",
    orientation="h",
    text="Score",
    title="Top 10 Countries with the Highest Happiness Score",
    labels={"Score": "Happiness Score", "Country or region": "Country"},
)

fig.update_traces(texttemplate="%{text:.2f}", textposition="outside")
fig.update_layout(width=800, height=500)
fig.show()

Interpretasi :   
- Top 10 didominasi negar negara eropa khususnya negera negara Nordik yang mendominasi puncak
- selisih score top 10 mempunyai selisih yang kecil , menunjukkan kesejateraan disana setara

## 5.Choropleth : Map Happiness Score

**Choropleth** adalah peta yang mewarnai tiap negara sesuai nilainya. Karena dataset memakai nama negara (bukan kode ISO), kita gunakan `locations="Country or region"` dengan `locationmode="country names"`.

### Grafik 3 - Choropleth world happiness score

In [43]:
df["Rank"]=(
    df["Score"].rank(ascending=False, method="min")
    .astype(int)
    .apply(lambda x: f"{x:02d}")
)

In [44]:
fig = px.choropleth(
    df,
    locations="Country or region",
    locationmode="country names",
    color="Score",
    hover_name="Country or region",
    hover_data={"Score": True, "Rank": True},
    color_continuous_scale="Viridis",
    title="World Happiness Score",
    labels={"Score": "Happiness Score"},
)

fig.update_layout(width=800, height=500)
fig.show()

Interpretasi :
- Pola Warna yang semakin terang, menunjukkan negara tersebut memiliki tingkat Happiness Score yang tinggi
- dapat diamati dibenua Afrika tidak ada warna yang terang, menunjukkan bahwa tingkat score happiness di benua ini masih berada di level menengah kebawah
- Kita bisa liat 1 per 1 negara didalam peta, sudah disedia keterangan ( Nama negara, Score Happiness, Urutan negara dalam Score Hapinnes )

## 6.Scatter Faktor Korupsi vs Kebebasan

Sekalian kita lihat hubungan dua faktor non-ekonomi, diwarnai berdasarkan skor.

### Grafik 4 -  Persepsi Korupsi vs Kebebasan

In [51]:
fig = px.scatter (
    df,
    x="Perceptions of corruption",
    y="Freedom to make life choices",
    color="Score",
    size= "Social support",
    color_continuous_scale="Viridis",
    hover_name="Country or region",
    hover_data={"Score": True, "Rank": True},
    title="Perceptions of Corruption vs Freedom to Make Life Choices",
    labels={
        "Perceptions of corruption": "Perceptions of Corruption",
        "Freedom to make life choices": "Freedom to Make Life Choices",
    },
)

fig.update_layout(width=800, height=500)
fig.show()


Interpretasi :   
- Negara dengan kebebasan tinggi dan persepsi korupsi tinggi ( masyarakat menilai pemerintah relatif bersih )
-  berwarna terang = skor tinggi
- Terdapat 2 anomali yang harus di selidiki, yaitu negara rwanda dan somalia yang memiliki kebebasan dan persepsi korupsi tinggi tetapi happiness scorenya berada di urutan 152 / 156 dan 112 / 156, Social support berpengaruh juga disini

## Kesimpulan & Insight

- **Plotly Express** membuat grafik interaktif (hover, zoom) hanya dengan beberapa baris kode.
- Kebahagiaan negara berkorelasi kuat dengan **GDP per kapita, dukungan sosial, dan harapan   hidup sehat**.
- **Negara Nordik** menempati puncak; pola geografis terlihat jelas lewat **choropleth**.
- Interaktivitas sangat membantu eksplorasi data multi-dimensi dan presentasi.

Plotly cocok dipakai saat kamu ingin pembaca **menjelajah** data sendiri.

## 💡 Insight & Jebakan Data

Grafik interaktif memberi pembaca kendali untuk **menjelajah** sendiri — sangat efektif untuk
presentasi dan dashboard yang dibagikan.

### ⚠️ Jebakan Data
- **Choropleth** mencocokkan negara lewat **nama**. Nama yang tak standar (mis. "United States"
  vs "USA") bisa **hilang dari peta** — periksa `locationmode`/kode ISO.
- Hubungan kuat GDP↗Score di peta **bukan bukti sebab-akibat**. Korelasi ≠ kausalitas
  (lihat catatan di NB04).